# 9. Stockage, sauvegarde et partage

Jusqu'ici, vous avez surtout **suivi le parcours** de l'atelier. Pour la suite, il faut vous préparer à l'**autonomie** : le balisage XML est un bac à sable, les scripts peuvent toucher des centaines de fichiers d'un coup, et une mauvaise manipulation sur la base SQLite se propage partout. Ce chapitre pose les réflexes de **stockage**, **sauvegarde** et **partage** avant que vous ne travailliez seul·e sur votre corpus.

## Deux sortes de dossiers

L'éditeur distingue ce que **vous** éditez (corpus, traductions) et ce que **l'application** maintient (cache, journaux, sauvegardes).

**1. Dossier de la base d'entités** (réglages de l'application)

C'est le dossier que vous avez choisi dans la configuration : il contient votre **base SQLite centrale** (`entities.sqlite`), les paquets d'autorités téléchargés et, le cas échéant, d'autres ressources réutilisées d'un projet à l'autre. C'est votre index personnel — le même 張衡, vos notes, vos concordances — pas le dossier que vous envoyez à un·e collègue.

**2. Dossiers de projet** (un corpus = un dossier)

Chaque projet regroupe vos transcriptions XML, vos fichiers de traduction, une **base SQLite du projet**, le schéma de validation et la mise en forme.

### Pourquoi deux bases, et comment elles se parlent

Les deux fichiers ne sont pas deux copies de la même chose : ils n'ont pas le même métier.

La **base du projet** voyage avec l'édition. Les attributs `@key` du XML pointent **toujours** vers elle. C'est elle que vous partagez (Git, copie du dossier) : un collaborateur peut ouvrir le corpus sans avoir votre base centrale. Sans ce fichier, les identifiants dans le texte ne mènent plus nulle part.

La **base centrale** est votre index de toute une carrière. Vous y désambiguïsez une personne une fois, puis vous la retrouvez dans d'autres projets. Elle reste **à vous**.

Le **pont** (panneau Entités → « Pont vers la base centrale ») relie les deux *pour vous* :

- **lier** une fiche du projet à une fiche déjà connue au centre ;
- **promouvoir** (ou synchroniser) une nouvelle personne vers le centre, pour la réutiliser ailleurs ;
- **trancher un conflit** si les deux côtés ont divergé — LJB n'écrase pas en silence.

Pourquoi ce découpage plutôt qu'une seule base ? Une base unique mélange vos notes privées et l'édition partageable, et deux personnes qui créent des fiches en même temps se marcheraient sur les identifiants. Une base par projet seulement vous forcerait à redésambiguïser les mêmes gens à chaque corpus. Un serveur en ligne serait un autre outil à administrer. Les deux fichiers, plus un pont optionnel, gardent l'édition autonome tout en vous laissant un cerveau personnel.

## Organisation typique d'un projet

À l'intérieur d'un dossier de projet, vous verrez notamment :

| Élément | Rôle |
|--------|------|
| **Fichiers `.xml`** à la racine du projet | Transcriptions balisées, métadonnées, `@key` vers les **entités du projet**. |
| **`entities.sqlite`** | Base d'entités **du projet** : fiches désambiguïsées et identifiants visés par les `@key`. |
| **Fichiers de traduction** | Texte traduit et notes, **séparés** de la source et reliés aux paragraphes par leurs identifiants (voir ch. 3). |
| **`schema/`** | Schéma RNG (par ex. TEI ALL), feuille CSS de surbrillance, etc. |
| **`.ljb/`** (caché) | Infrastructure de l'éditeur : cache des autorités, suggestions, journal de décisions. **Ne pas éditer à la main** ; ne pas supprimer sans savoir pourquoi. |
| **`.leafwriter-time-machine/`** (caché) | Instantanés du projet (voir ci-dessous). |

### Tout doit voyager ensemble

Les fichiers ne sont pas indépendants :

- les **traductions** pointent vers des paragraphes précis du XML source ;
- les attributs **`key`** dans le texte pointent vers des fiches dans la **base SQLite du projet** (pas vers la base centrale).

Si vous copiez une transcription sans sa traduction, ou sans `entities.sqlite` du projet, les panneaux se vident, les liens cassent et vos statistiques (ch. 8) deviennent incohérentes. **Sauvegardez et synchronisez l'ensemble du dossier de projet**, pas un fichier isolé. La base centrale, elle, n'a pas à voyager avec l'équipe.

L'éditeur vous protège en partie (fichiers sous `.ljb/` exclus du corpus pour les scripts), mais rien n'empêche de modifier la base SQLite avec un éditeur externe ou un script mal ciblé — d'où l'importance des copies de sécurité.

## Sauvegarde : Time Machine

Vous ferez un jour une **grosse** erreur : remplacement global d'un nom, regex trop greedy, propagation de mauvaises clés sur des milliers de mentions. Les XML restent légers ; profitez-en pour prendre des **instantanés** réguliers.

Dans le menu du projet, **Time Machine** enregistre un snapshot du dossier de projet (hors dossiers de sauvegarde eux-mêmes). Après la bêtise, vous pouvez **restaurer** un état antérieur de l'ensemble des fichiers du projet.

Ce n'est pas un substitut à une copie sur disque externe ou dans le cloud : c'est votre filet de sécurité **à chaud**, sur la machine où vous travaillez.

## Synchronisation cloud (Dropbox, iCloud, etc.)

Mettre **tout le dossier de projet** dans le cloud est un bon moyen de le sauvegarder, de le retrouver sur une autre machine, et de le partager avec un·e collègue. Les XML et `entities.sqlite` du projet **doivent rester ensemble** : ce n'est pas le corpus d'un côté et la base de l'autre.

**Cependant**, la sync peut interférer avec l'écriture SQLite (fichiers annexes `-wal` / `-shm`) et **corrompre la base**. Si vous acceptez ce risque : sauvegardez souvent (Time Machine, copies à froid), attendez la fin d'une sync, et **fermez LJB** avant d'ouvrir le même projet sur un autre appareil.

Le même risque vaut pour la **base centrale** — en plus grave, car elle accumule le travail de désambiguïsation de tous vos projets, pas seulement d'une édition. Ne la mettez dans le cloud que si vous avez des copies indépendantes ; pour plusieurs de *vos* machines, la synchronisation d'entités dans les réglages (Profil) est plus sûre qu'un dossier Dropbox « vivant ».

Attention : la sync cloud **répercute aussi les suppressions**. Ce n'est pas une archive à long terme ; combinez-la avec Time Machine ou Git.

## GitHub (historique et partage)

Dropbox et iCloud copient le dossier **en continu**, sans vous demander. **Git** fait autre chose : c'est un carnet de versions. Vous décidez quand enregistrer un **jalon** (« 12 juin, fin du juan 3 ») ; chaque jalon a un message, et vous pouvez revenir à n'importe lequel plus tard. **GitHub** est un site (gratuit, avec des dépôts privés) qui **héberge une copie** de ce carnet : sauvegarde hors machine, travail à deux, éventuellement publication.

LJB **n'est pas** Git : il ouvre et enregistre des fichiers locaux. Git et GitHub vivent à côté. Pour éviter la ligne de commande, [GitHub Desktop](https://github.com/apps/desktop) suffit : une fenêtre, des boutons, le même dossier de projet.

### Pourquoi c'est une bonne solution ici

- **Vous choisissez le moment.** Un jalon, ce n'est pas « le fichier tel qu'il flotte dans le cloud à 14 h 37 », c'est un état que vous avez jugé digne d'être conservé.
- **L'historique se lit.** Pour le XML, GitHub montre souvent le changement **ligne par ligne**. Time Machine restaure un dossier entier ; Git raconte *ce qui* a changé.
- **Le dossier reste ensemble.** On versionne le **projet** (transcriptions + `entities.sqlite` + traductions). Un·e collègue clone le même dossier et ouvre LJB ; votre base centrale, elle, reste chez vous.
- **Publication.** Un dépôt privé le reste tant que vous le souhaitez. Le passer en public, c'est exposer le corpus (droits d'auteur et données personnelles d'abord) sans tout réorganiser.

### Les inconvénients (à connaître avant)

- **Ce n'est pas automatique.** Oublier d'envoyer un jalon, c'est n'avoir **aucune** copie sur GitHub. Dropbox, lui, part tout seul — avec les risques SQLite vus plus haut.
- **Le vocabulaire déroute** la première heure : *dépôt* (le projet versionné), *commit* (un jalon local), *push* (l'envoyer sur GitHub), *pull* (récupérer ce qui est sur GitHub). Après, c'est toujours les mêmes quatre gestes.
- **Les conflits.** Si deux personnes modifient la **même ligne** du même fichier sans s'être d'abord mises à jour, Git s'arrête et demande de trancher à la main. C'est formateur, mais désagréable la première fois. D'où la règle : *pull* avant de travailler, *push* quand un jalon est prêt, et éviter d'éditer les entités à deux en même temps.
- **La base SQLite est opaque pour Git.** Il en conserve des versions, mais ne peut pas afficher « telle fiche personne a changé » comme pour le XML. Un commit de `entities.sqlite`, c'est souvent « la base a bougé », pas un diff lisible.
- **Un dépôt public se voit.** Vérifiez la liste des fichiers avant le premier envoi : jamais de clé API, de textes strictement privés, ni de données personnelles. La base **centrale** n'a en général **pas** sa place dans ce dépôt.

### Premiers gestes (dans GitHub Desktop)

1. Créer un compte GitHub, installer Desktop, créer un dépôt **privé** (si le corpus n'est pas destiné à être public).
2. L'associer à votre **dossier de projet** (pas au dossier de la base centrale).
3. Ajouter un fichier **`.gitignore`** : une liste de noms que Git doit ignorer, par exemple `.ljb/`, `.leafwriter-time-machine/`, caches et exports régénérables. Sans cela, le carnet se remplirait de fichiers inutiles.
4. **Commit** (jalon sur votre machine) puis **Push** (copie sur GitHub).
5. **Pull** avant de reprendre le travail sur un autre ordinateur, ou avec un·e collègue.

Entraînez-vous d'abord sur une **copie** ou un projet d'essai, pas sur le seul exemplaire de votre corpus.

## Sauvegardes cloud et synchronisation d'entités (LJB)

Dropbox copie des **fichiers**. Git enregistre des **jalons** du dossier de projet. Pour la **base d'entités** — surtout la base centrale, qui accumule des années de désambiguïsation — LJB propose une troisième voie, conçue pour SQLite : des **sauvegardes périodiques** hors machine, et une **synchronisation fiche par fiche** vers une base distante. Le fichier vivant reste sur **votre disque** ; le cloud n'essaie pas de le recopier pendant que LJB l'écrit.

Deux services distincts, dans **Réglages → Profil** :

| | **Sauvegarde cloud** | **Synchronisation entre machines** |
|--|----------------------|-------------------------------------|
| À quoi ça sert | Filet de sécurité : copies compressées à intervalle régulier et à la fermeture de LJB. En cas de perte ou de corruption, vous **restaurez** un instantané (cela *remplace* la base locale). | Garder la même base d'entités à jour sur **plusieurs de vos ordinateurs**. Les fiches voyagent une par une ; en cas de divergence, LJB vous demande de trancher (comme le pont), au lieu d'écraser en silence. |
| Ce que ce n'est pas | Ce n'est **pas** un travail à deux sur le même corpus (ça, c'est le dossier de projet + Git). | Ce n'est **pas** une archive « au cas où » : une sync fusionne l'état courant, elle ne conserve pas dix vieux instantanés. |

Les deux se complètent : la sync pour le quotidien entre machines ; la sauvegarde pour le jour où le disque meurt.

### Où l'héberger

Le protocole n'est pas lié à un seul prestataire.

- **[Cloudflare](https://www.cloudflare.com/)** — vous pouvez le mettre en place **gratuitement** (compte gratuit ; les quotas du plan gratuit suffisent largement pour une base d'historien). C'est la voie documentée aujourd'hui : stockage d'instantanés + base distante pour la sync. Un assistant dans les réglages (Profil) indique les champs à remplir une fois le compte créé.
- **[Huma-Num](https://www.huma-num.fr/)** — la TGIR française pour les humanités numériques peut héberger le **même** dispositif (sauvegardes + synchronisation vers une base externe). Utile si vous préférez une infrastructure de recherche plutôt qu'un compte cloud grand public, ou si votre laboratoire y a déjà un espace.

Dans les deux cas, les identifiants restent **à vous** ; LJB ne publie pas votre base d'entités.

Ce n'est pas obligatoire le premier jour. Time Machine + copies du dossier de projet suffisent pour l'atelier. Dès que la base centrale compte vraiment, activez au moins la **sauvegarde cloud**.

## Préparation

- [ ] Repérer dans l'explorateur de fichiers votre dossier de **base centrale** et un dossier de projet (avec son `entities.sqlite`).
- [ ] Dans le panneau Entités, ouvrir le **pont** et voir ce qui est lié, à synchroniser, ou en conflit.
- [ ] Créer un instantané Time Machine de test, puis vérifier qu'il apparaît dans la liste.
- [ ] (Recommandé) Initialiser un dépôt Git ou GitHub Desktop sur une **copie** ou un projet d'essai, avec un `.gitignore` adapté.
- [ ] (Quand la base centrale commencera à compter) Ouvrir **Réglages → Profil** et voir les panneaux *Sauvegarde cloud* et *Synchronisation entre machines* — sans obligation de tout activer le jour même.
- [ ] Lister trois fichiers **irremplaçables** (à sauvegarder tels quels) et trois **regénérables** (exports, caches).

Le chapitre suivant (« Validation XML ») reprend la récupération lorsque, malgré tout, quelque chose se brise.
